<a href="https://colab.research.google.com/github/Limeng-svg/Grounded-PPE-Safety-Copilot/blob/main/notebooks/02_yoloworld_mini_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This part is mainly for small range of validation set

In [2]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Grounded-PPE-Safety-Copilot"
)

IMAGE_DIR = PROJECT_ROOT / "data" / "mini_eval" / "images"
LABEL_DIR = PROJECT_ROOT / "data" / "mini_eval" / "labels"

for directory in [IMAGE_DIR, LABEL_DIR]:
    directory.mkdir(parents = True, exist_ok = True)

print("Image store place is: ", IMAGE_DIR)

Mounted at /content/drive
Image store place is:  /content/drive/MyDrive/Grounded-PPE-Safety-Copilot/data/mini_eval/images


Now we need ***at least 12 pictures*** to do a more clear validation

In [3]:
from PIL import Image
import hashlib

extensions = {".jpg", ".jpeg", ".png", ".webq"}
paths = sorted(
    p for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() in extensions
)

seen = {}
valid = []
problems = []

for path in paths:
  if "_prediction" in path.stem.lower():
    problems.append(f"Please remove from prediction paragraph: {path.name}")
    continue

  try:
    with Image.open(path) as img:
      width, height = img.size
      img.verify()

    digest = hashlib.sha256(path.read_bytes()).hexdigest()

    if digest in seen:
      problems.append(f"Duplicate image: {path.name} and {seen[digest]}")
      continue
    seen[digest] = path.name
    valid.append(path)
    print(f"{path.name} | {width} × {height}")

  except Exception as error:
        problems.append(f"Cant upload: {path.name} | {error}")

print(f"\nValid and the content not duplicated: {len(valid)} pages")

for problem in problems:
    print("Need progress: ", problem)

if problems:
    print("Please handle the issues above first; the code won’t automatically delete files.")
elif len(valid) < 12:
    print(f"At least complement {12 - len(valid)} pages.")
else:
    print("The images are ready, you can move on to the bounding box annotation stage.")

easy_01.jpg | 4000 × 6000
easy_02.jpg | 6000 × 4000
easy_03.jpg | 2624 × 3936
easy_04.jpg | 4000 × 6000
hard_01.jpg | 2356 × 4000
hard_02.jpg | 5568 × 3712
hard_03.jpg | 5562 × 3685
hard_04.jpg | 6240 × 4160
violation_01.jpg | 1024 × 706
violation_02.jpg | 5472 × 3648
violation_03.jpg | 6720 × 4480
violation_04.jpg | 2614 × 3267

Valid and the content not duplicated: 12 pages
The images are ready, you can move on to the bounding box annotation stage.


In [4]:
from google.colab import drive, files
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
from hashlib import sha256
from collections import Counter

drive.mount('/content/drive')

DATA_DIR = Path("/content/drive/MyDrive/Grounded-PPE-Safety-Copilot/data/mini_eval")
IMAGE_DIR = DATA_DIR / "images"
LABEL_DIR = DATA_DIR / "labels"
CLASS_NAMES = ["person", "hard_hat", "safety_vest"]

assert IMAGE_DIR.is_dir(), "Cant find the directory, please check the drive path"

images = [p for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webq"}]

expected = {f"{p.stem}.txt" for p in images}

assert len(images) == len(expected) == 12, "The number of images or file names does not match expectations."

uploaded = files.upload()
assert len(uploaded) == 1, "Please only uploading lastest ZIP."
zip_bytes = next(iter(uploaded.values()))

approved_hash = "48920aec182d79b58f33c9745928516beca597cdc1d2b7b9b8364bfb675a78f7"
assert sha256(zip_bytes).hexdigest() == approved_hash, "It is not the ZIP for this time, choose another one."

with ZipFile(BytesIO(zip_bytes)) as archive:
  assert len(archive.namelist()) == 12
  assert set(archive.namelist()) == expected, "Label filename can not match the picture."
  labels = {name: archive.read(name).decode("utf-8") for name in sorted(expected)}

LABEL_DIR.mkdir(parents=True, exist_ok=True)

# Check the conflict firstly, avoid covering current different version
for name, content in labels.items():
  destination = LABEL_DIR / name
  if not destination.exists():
    destination.write_text(content, encoding="utf-8")

counts = Counter(
    int(line.split()[0])
    for content in labels.values()
    for line in content.splitlines()
    if line.strip()
)

print(f"import is finished: {len(labels)} label files, {sum(counts.values())} frames")
for class_id, name in enumerate(CLASS_NAMES):
    print(f"{name}: {counts[class_id]}")
print("Store location", LABEL_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving labels_my-project-name_2026-08-26-04-34-06.zip to labels_my-project-name_2026-08-26-04-34-06.zip
import is finished: 12 label files, 150 frames
person: 60
hard_hat: 53
safety_vest: 37
Store location /content/drive/MyDrive/Grounded-PPE-Safety-Copilot/data/mini_eval/labels


In [5]:
%pip -q install "ultralytics==8.4.129" "git+https://github.com/ultralytics/CLIP.git@68dce32140994dfcb645a1320c4ebdc034fc19fd"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.0/66.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00


Prepare for **evaluation set**

In [6]:
from google.colab import drive, files
from pathlib import Path
from collections import Counter
import tempfile, shutil, json
import yaml, torch, ultralytics

drive.mount('/content/drive')

assert ultralytics.__version__ == "8.4.129", "launch the block after rerun."
assert torch.cuda.is_available(), "Set the core to T4 GPU."
print("GPU:",torch.cuda.get_device_name(0))

ROOT = Path("/content/drive/MyDrive/Grounded-PPE-Safety-Copilot")
SOURCE = ROOT / "data/mini_eval"

# label: 0 = person 1=hard_hat 2=vest
CLASS_NAMES = ["person", "hard_hat", "safety_vest"]
PROMPTS = ["person", "hard hat", "safety vest"]

images = sorted((SOURCE / "images").glob("*.jpg"))
expected = {
    f"{group}_{i:02d}"
    for group in ["easy", "hard", "violation"]
    for i in range(1, 5)
}
assert len(images) == 12 and {p.stem for p in images} == expected, \
    "图片文件名或数量不符合预期。"

LOCAL = Path(tempfile.mkdtemp(prefix="ppe_mini_eval_", dir="/content"))
for folder in ["images/val", "labels/val"]:
    (LOCAL / folder).mkdir(parents=True)

counts = Counter()

for img in images:
    label = SOURCE / "labels" / f"{img.stem}.txt"
    assert label.is_file(), f"缺少标注：{label.name}"

    rows = label.read_text(encoding="utf-8").splitlines()
    counts.update(int(row.split()[0]) for row in rows if row.strip())

    shutil.copy2(img, LOCAL / "images/val" / img.name)
    shutil.copy2(label, LOCAL / "labels/val" / label.name)

assert counts == Counter({0: 60, 1: 53, 2: 37}), \
    f"标注数量不符合已审核版本：{counts}"

# YAML 中的名字必须与模型提示词完全一致
DATA_YAML = LOCAL / "mini_eval.yaml"
DATA_YAML.write_text(
    yaml.safe_dump({
        "path": str(LOCAL),
        "train": None,          # 本轮只验证，不训练
        "val": "images/val",
        "names": dict(enumerate(PROMPTS)),
    }, sort_keys=False),
    encoding="utf-8"
)

print("数据准备完成：12 张图片，150 个真实框。")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4
数据准备完成：12 张图片，150 个真实框。
